<a href="https://colab.research.google.com/github/DonyaRamezani/RUL_LSTM_with_Uncertainty/blob/main/rul_lstm_with_uncertainty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, sys, traceback
import math, random
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

# PyTorch

In [2]:
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn.functional as F

# Optional notebook display

In [3]:
try:
    from caas_jupyter_tools import display_dataframe_to_user
    _HAS_DISPLAY = True
except Exception:
    _HAS_DISPLAY = False

# Utils

In [4]:
def sanitize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    cols = []
    for c in df.columns:
        if isinstance(c, tuple):
            cols.append("_".join([str(x) for x in c]))
        else:
            cols.append(str(c))
    df = df.copy()
    df.columns = cols
    return df

# Synthetic dataset

In [5]:
def generate_synthetic_component_run(seq_len=200, n_sensors=3, failure_time=None, seed=None):
    if seed is not None:
        np.random.seed(seed)
    if failure_time is None:
        failure_time = np.random.randint(int(seq_len * 0.6), seq_len)
    times = np.arange(seq_len)
    RUL = np.clip(failure_time - times, 0, None)
    data = {}
    for s in range(n_sensors):
        baseline = 10 + s * 2
        drift = 0.01 * times * (1 + 0.5 * np.random.randn())
        cyclic = 0.5 * np.sin(2 * np.pi * times / (30 + 5 * np.random.randn()))
        degradation = (times / failure_time) ** (1.5 + 0.5 * np.random.randn()) * 3.0
        noise = np.random.normal(scale=0.3 + 0.1 * s, size=seq_len)
        data[f"sensor_{s}"] = baseline + drift + cyclic + degradation + noise
    df = pd.DataFrame(data)
    df["time"] = times
    df["RUL"] = RUL
    df["failure_time"] = failure_time
    return df

def create_synthetic_dataset(n_components=200, seq_len=200, n_sensors=3, seed=42):
    np.random.seed(seed)
    runs = []
    for i in range(n_components):
        ft = np.random.randint(int(seq_len * 0.5), seq_len)
        df = generate_synthetic_component_run(seq_len=seq_len, n_sensors=n_sensors, failure_time=ft)
        df["component_id"] = int(i)
        runs.append(df)
    big = pd.concat(runs, ignore_index=True)
    big = sanitize_column_names(big)
    big["component_id"] = big["component_id"].astype(int)
    big["time"] = big["time"].astype(int)
    return big

# Noise / missing values

In [6]:
def introduce_missing_and_noisy(df, missing_rate=0.05, drop_random_runs=0.02, seed=1):
    np.random.seed(seed)
    df2 = df.copy().reset_index(drop=True)
    df2 = sanitize_column_names(df2)
    sensor_cols = [c for c in df2.columns if c.startswith("sensor_")]
    # random NaNs
    for col in sensor_cols:
        mask = np.random.rand(len(df2)) < missing_rate
        df2.loc[mask, col] = np.nan
    return df2

# Preprocess

In [7]:
from sklearn.impute import KNNImputer
from scipy.signal import welch

def preprocess_run(df_run, sensor_cols, window_size=5):
    d = df_run.copy().reset_index(drop=True)
    d[sensor_cols] = d[sensor_cols].ffill().bfill()
    imputer = KNNImputer(n_neighbors=3)
    d[sensor_cols] = imputer.fit_transform(d[sensor_cols])
    for col in sensor_cols:
        d[f"{col}_ma"] = d[col].rolling(window=window_size, min_periods=1).mean()
        d[f"{col}_std"] = d[col].rolling(window=window_size, min_periods=1).std().fillna(0.0)
    for col in sensor_cols:
        try:
            f, Pxx = welch(d[col], nperseg=min(64, max(4, len(d[col]))))
            centroid = (f * Pxx).sum() / (Pxx.sum() + 1e-9)
        except Exception:
            centroid = 0.0
        d[f"{col}_spec_centroid"] = centroid
    return d

# Dataset / Model

In [8]:
SEQ_IN = 40
SEQ_STRIDE = 5

class SequenceDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences
    def __len__(self): return len(self.sequences)
    def __getitem__(self, idx):
        s = self.sequences[idx]
        return torch.from_numpy(s["X"]), torch.tensor([s["y"]], dtype=torch.float32)

class RULLSTM(nn.Module):
    def __init__(self, n_features, hidden_size=64, n_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden_size, num_layers=n_layers,
                            batch_first=True, dropout=dropout)
        self.drop = nn.Dropout(p=dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        last = self.drop(last)
        return self.fc(last).squeeze(1)

# Uncertainty estimation with MC Dropout

In [9]:
def enable_dropout(model):
    for m in model.modules():
        if m.__class__.__name__.startswith("Dropout"):
            m.train()

def predict_with_uncertainty(model, loader, device, n_mc=50):
    model.eval()
    enable_dropout(model)
    all_preds_mc = []
    all_targets = []
    with torch.no_grad():
        for mc in range(n_mc):
            batch_preds = []
            for Xb, yb in loader:
                Xb = Xb.to(device)
                pv = model(Xb).cpu().numpy()
                batch_preds.append(pv)
                if mc == 0:
                    all_targets.append(yb.squeeze(1).numpy())
            all_preds_mc.append(np.concatenate(batch_preds, axis=0))
    preds_stack = np.vstack(all_preds_mc)
    preds_mean = preds_stack.mean(axis=0)
    preds_std = preds_stack.std(axis=0, ddof=0)
    targets = np.concatenate(all_targets, axis=0)
    return preds_mean, preds_std, targets

# Scenario-based decision (stochastic programming)

In [10]:
def sample_rul_scenarios(pred, sigma, n_scen=100, trunc_min=0.0):
    sigma = max(1.0, sigma)
    samples = np.random.normal(loc=pred, scale=sigma, size=n_scen)
    samples = np.clip(samples, trunc_min, None)
    return samples

def cost_for_action_in_scenario(rul_scenario, action, params):
    C_r = params.get("C_r", 500.0)
    C_d = params.get("C_d", 1000.0)
    C_f = params.get("C_f", 10000.0)
    H = params.get("H", 30.0)
    p_fail = 1.0 if rul_scenario <= H else 0.0
    if action == "Repair":
        cost = C_r + p_fail * 0.1 * C_f
    elif action == "Reuse":
        cost = p_fail * C_f
    elif action == "Discard":
        cost = C_d
    else:
        raise ValueError("Unknown action")
    return cost

def scenario_decision(pred, sigma, params, n_scen=200):
    scenarios = sample_rul_scenarios(pred, sigma, n_scen=n_scen, trunc_min=0.0)
    actions = ["Repair", "Reuse", "Discard"]
    exp_cost = {}
    for a in actions:
        costs = [cost_for_action_in_scenario(r, a, params) for r in scenarios]
        exp_cost[a] = float(np.mean(costs))
    best_action = min(exp_cost, key=exp_cost.get)
    return best_action, exp_cost, scenarios

# Main

In [11]:
def run_demo():
    try:
        print("Generating synthetic dataset...")
        df = create_synthetic_dataset(n_components=200, seq_len=200, n_sensors=3)
        print("Introducing missing values and noise...")
        df_imperfect = introduce_missing_and_noisy(df, missing_rate=0.03, drop_random_runs=0.02)

        sensor_cols = [c for c in df_imperfect.columns if c.startswith("sensor_")]
        print(f"Detected sensor columns: {sensor_cols}")

        print("Preprocessing runs (data-centric)...")
        processed_runs = []
        for cid, grp in tqdm(df_imperfect.groupby("component_id")):
            processed = preprocess_run(grp, sensor_cols, window_size=8)
            processed["component_id"] = int(cid)
            processed_runs.append(processed)
        df_proc = pd.concat(processed_runs, ignore_index=True)

        # features
        feature_cols = []
        for c in sensor_cols:
            feature_cols += [c, f"{c}_ma", f"{c}_std", f"{c}_spec_centroid"]

        # sequences
        sequences = []
        for cid, grp in tqdm(df_proc.groupby("component_id")):
            g = grp.reset_index(drop=True)
            n = len(g)
            for start in range(0, n - SEQ_IN + 1, SEQ_STRIDE):
                window = g.iloc[start:start+SEQ_IN]
                X = window[feature_cols].values.astype(np.float32)
                y = float(window["RUL"].values[-1])
                sequences.append({"X": X, "y": y, "component_id": int(cid)})

        dataset = SequenceDataset(sequences)
        train_size = int(len(dataset) * 0.8)
        val_size = int(len(dataset) * 0.1)
        test_size = len(dataset) - train_size - val_size
        train_ds, val_ds, test_ds = random_split(dataset, [train_size, val_size, test_size])
        train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

        n_features = len(feature_cols)
        model = RULLSTM(n_features=n_features, hidden_size=64, n_layers=2, dropout=0.2)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        criterion = nn.MSELoss()

        # training
        def train_model(model, train_loader, val_loader, epochs=8):
            for epoch in range(epochs):
                model.train()
                train_losses = []
                for Xb, yb in train_loader:
                    Xb, yb = Xb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    preds = model(Xb)
                    loss = criterion(preds, yb.squeeze(1))
                    loss.backward()
                    optimizer.step()
                    train_losses.append(loss.item())
                val_losses = []
                model.eval()
                with torch.no_grad():
                    for Xv, yv in val_loader:
                        Xv, yv = Xv.to(device), yv.to(device)
                        pv = model(Xv)
                        val_losses.append(criterion(pv, yv.squeeze(1)).item())
                print(f"Epoch {epoch+1}/{epochs} — train_loss: {np.mean(train_losses):.4f} — val_loss: {np.mean(val_losses):.4f}")
        print("Training LSTM model...")
        train_model(model, train_loader, val_loader, epochs=8)

        # === Uncertainty estimation ===
        print("Estimating uncertainty with MC Dropout...")
        preds_mean, preds_std, targets = predict_with_uncertainty(model, test_loader, device, n_mc=60)

        print("Sample predictions (mean ± std):")
        for i in range(8):
            print(f"pred_mean={preds_mean[i]:6.2f}  sigma={preds_std[i]:5.2f}  target={targets[i]:6.2f}")

        # === Scenario-based decision ===
        params = {"C_r": 500.0, "C_d": 1000.0, "C_f": 10000.0, "H": 30.0}
        print("\nScenario-based decisions on first 10 items:")
        for i, (p, s) in enumerate(zip(preds_mean[:10], preds_std[:10])):
            action, costs, sc = scenario_decision(float(p), float(s), params, n_scen=300)
            print(f"Item {i}: pred={p:.1f}, sigma={s:.2f} -> {action}, costs={costs}")

        print("Pipeline finished successfully. Artifacts saved under ./artifacts/")
    except Exception as err:
        tb = traceback.format_exc()
        with open("error_traceback.txt", "w") as f:
            f.write(tb)
        print("Error! see error_traceback.txt")
        raise

if __name__ == "__main__":
    run_demo()

Generating synthetic dataset...
Introducing missing values and noise...
Detected sensor columns: ['sensor_0', 'sensor_1', 'sensor_2']
Preprocessing runs (data-centric)...


100%|██████████| 200/200 [00:02<00:00, 91.42it/s]


Training LSTM model...
Epoch 1/8 — train_loss: 3140.6118 — val_loss: 2563.0520
Epoch 2/8 — train_loss: 2170.9468 — val_loss: 1798.4169
Epoch 3/8 — train_loss: 1785.0080 — val_loss: 1759.3415
Epoch 4/8 — train_loss: 1763.4796 — val_loss: 1761.9129
Epoch 5/8 — train_loss: 1775.8598 — val_loss: 1759.5347
Epoch 6/8 — train_loss: 1780.8048 — val_loss: 1758.9049
Epoch 7/8 — train_loss: 1633.6452 — val_loss: 1186.2832
Epoch 8/8 — train_loss: 630.5542 — val_loss: 368.6260
Estimating uncertainty with MC Dropout...
Sample predictions (mean ± std):
pred_mean= 71.03  sigma= 4.38  target= 84.00
pred_mean= 65.70  sigma= 3.90  target= 12.00
pred_mean=  3.77  sigma= 0.90  target=  0.00
pred_mean= 14.95  sigma= 1.49  target= 26.00
pred_mean=  7.18  sigma= 1.29  target=  4.00
pred_mean=  4.18  sigma= 1.14  target=  0.00
pred_mean=  4.46  sigma= 1.10  target=  0.00
pred_mean= 68.13  sigma= 4.03  target= 50.00

Scenario-based decisions on first 10 items:
Item 0: pred=71.0, sigma=4.38 -> Reuse, costs={'Rep